## Load cleaned data

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

data = pd.read_csv("outputs/cleaned_data.csv", parse_dates=["Order Date"])
data.shape

(1452, 9)

In [11]:
data.head()

,Order Date,Sales,year,month,day,day_of_week,quarter,sales_lag1,rolling_mean_7
0,2014-01-09,40.544,2014,1,9,3,1,0.000,694.120857
1,2014-01-10,54.830,2014,1,10,4,1,40.544,699.604000
2,2014-01-11,9.940,2014,1,11,5,1,54.830,659.872571
3,2014-01-12,0.000,2014,1,12,6,1,9.940,657.081714
4,2014-01-13,3553.795,2014,1,13,0,1,0.000,535.181000


## Train test split

In [12]:
features = ["year", "month", "day", "day_of_week", "quarter", "sales_lag1", "rolling_mean_7"]
target = "Sales"

split_index = int(len(data) * 0.8)
train = data.iloc[:split_index]
test = data.iloc[split_index:]

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

print(X_train.shape, X_test.shape)

(1161, 7) (291, 7)


## Linear regression baseline

In [13]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

In [14]:
lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = mean_squared_error(y_test, lr_preds) ** 0.5
print("linear regression")
print("MAE:", lr_mae)
print("RMSE:", lr_rmse)

linear regression
MAE: 1649.9155703855993
RMSE: 2265.7430175580553


## Random forest model

In [15]:
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

In [16]:
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = mean_squared_error(y_test, rf_preds) ** 0.5
print("random forest")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)

random forest
MAE: 1658.9240478917525
RMSE: 2333.872830740519


## Compare results

In [17]:
results = pd.DataFrame({
    "model": ["Linear Regression", "Random Forest"],
    "MAE": [lr_mae, rf_mae],
    "RMSE": [lr_rmse, rf_rmse]
})
results

,model,MAE,RMSE
0,Linear Regression,1649.915570,2265.743018
1,Random Forest,1658.924048,2333.872831


## Save predictions

In [18]:
output = test[["Order Date", "Sales"]].copy()
output["predicted_sales"] = rf_preds
output.to_csv("outputs/predictions.csv", index=False)

## Forecast next 30 days

In [19]:
final_model = RandomForestRegressor(n_estimators=200, random_state=42)
final_model.fit(data[features], data[target])

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
history = data[["Order Date", "Sales"]].copy()
future_days = 30
future_rows = []

last_date = history["Order Date"].max()

for i in range(future_days):
    next_date = last_date + pd.Timedelta(days=1)

    lag1 = history["Sales"].iloc[-1]
    rolling_mean_7 = history["Sales"].iloc[-7:].mean()

    row = {
        "year": next_date.year,
        "month": next_date.month,
        "day": next_date.day,
        "day_of_week": next_date.dayofweek,
        "quarter": (next_date.month - 1) // 3 + 1,
        "sales_lag1": lag1,
        "rolling_mean_7": rolling_mean_7
    }

    pred = final_model.predict(pd.DataFrame([row])[features])[0]

    future_rows.append({"Order Date": next_date, "predicted_sales": pred})
    history = pd.concat([history, pd.DataFrame([{"Order Date": next_date, "Sales": pred}])], ignore_index=True)
    last_date = next_date

future_df = pd.DataFrame(future_rows)
future_df.head()

,Order Date,predicted_sales
0,2017-12-31,1729.624161
1,2018-01-01,2349.445195
2,2018-01-02,2125.732820
3,2018-01-03,677.858608
4,2018-01-04,2923.059140


In [21]:
future_df.to_csv("outputs/future_predictions.csv", index=False)